# 11 하이브리드 수요예측 — SBC vs ML scheme 비교

**2-type(고변동 E · 저변동 C)** 에 대해, 10장과 동일한 **LSTM + Best 임베딩** 예측으로
**SBC(rule-base) vs ML(AE+KMeans) 클러스터링 scheme**을 **제품수 가중 WMAPE**로 비교합니다.

## 논문(§4.5·5.3)과의 대응
| 논문 (Daiso 2센터) | 본 실습 (Ecuador 2-type) |
|---|---|
| Center A (저변동) → **ML** 우세 | type **C** (저변동, System CV 최저) |
| Center B (고변동) → **SBC** 우세 | type **E** (고변동, System CV 최고) |
| WMAPE = Σ n_k·MAPE_k / Σ n_k | 동일 |

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.experiment_data import load_forecast_frames

df, feat_df = load_forecast_frames()
phase2 = pd.read_parquet(DATA_PROCESSED / 'phase2_results.parquet')
phase2_best = pd.read_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv')
print('Phase2 (XGBoost×6임베딩) | rows:', len(phase2), '| 조건:', len(phase2_best))


Phase2 (XGBoost×6임베딩) | rows: 792 | 조건: 12


### ① 가중 WMAPE — type별 SBC vs ML

In [2]:
import numpy as np
from utils.phase_analysis import validation_weights, build_family_from_phase2_best, thesis_wmape_by_type
from utils.stats_summary import type_variation_table
from utils.config import selected_type_list

# 조건별 XGBoost + Best 임베딩(10장) 제품 단위 결과
val_weights = validation_weights(df)
family_final = build_family_from_phase2_best(phase2, phase2_best, val_weights)

# 논문식 WMAPE = Σ n_k·MAPE_k / Σ n_k (클러스터 제품수 가중, §4.5 Eq.50)
type_wmape = thesis_wmape_by_type(family_final)
print('=== type별 SBC vs ML (제품수 가중 WMAPE) ===')
display(type_wmape)
print('scheme 우세:', type_wmape['better_scheme'].value_counts().to_dict())

# System-Level CV(고/저변동 판별 지표)와 대조
var = type_variation_table(df)[['sys_cv', 'sku_mean_cv']].round(3)
sel = selected_type_list()  # [고변동, 저변동]
out = type_wmape.join(var)
out['variation'] = np.where(out.index == sel[0], 'high(고변동)',
                    np.where(out.index == sel[1], 'low(저변동)', ''))
print('=== scheme 우세 × System-Level CV ===')
display(out[['variation', 'sys_cv', 'SBC_wmape', 'ML_wmape', 'better_scheme']])

=== type별 SBC vs ML (제품수 가중 WMAPE) ===


,SBC_wmape,ML_wmape,delta_SBC_minus_ML,better_scheme
type,,,,
C,38.72,38.21,0.51,ML
E,46.64,46.87,-0.23,SBC


scheme 우세: {'ML': 1, 'SBC': 1}
=== scheme 우세 × System-Level CV ===


,variation,sys_cv,SBC_wmape,ML_wmape,better_scheme
type,,,,,
C,low(저변동),0.259,38.72,38.21,ML
E,high(고변동),0.429,46.64,46.87,SBC


### ② 해석 — SBC vs ML scheme (제품수 가중 WMAPE)

**WMAPE = Σ_k n_k·MAPE_k / Σ_k n_k** (클러스터 제품수 가중, 논문 §4.5 Eq.50) · base = **LSTM + Best 임베딩**

#### 결과 — 논문 가설과 방향 일치 ✅
| type | System CV | SBC WMAPE | ML WMAPE | 우세 | 논문 가설 |
|------|-----------|-----------|----------|------|-----------|
| **C** (저변동) | 0.259 | 38.72 | **38.21** | **ML** | 저변동→ML ✅ |
| **E** (고변동) | 0.429 | **46.64** | 46.87 | **SBC** | 고변동→SBC ✅ |

→ **저변동 C는 ML, 고변동 E는 SBC**가 우세 — 논문(저변동 센터 A→ML, 고변동 센터 B→SBC)의 **변동성↔scheme 방향과 일치**.

#### LSTM base가 만든 차이 (중요)
- **XGBoost base였을 때**는 ML 군집의 롱테일 62개를 한 패널에 묶어 예측이 폭주 → ML이 크게 뒤져 **둘 다 SBC 우세**였음.
- **robust한 LSTM base**(mean MAPE 최저)로 바꾸니 롱테일도 안정적으로 예측 → **ML scheme이 경쟁력을 회복**하고, 저변동 C에서 ML이 앞서며 논문 방향과 정합.
- 즉 scheme 우열은 **base 모델의 robust성**에 민감하며, robust한 신경망 base에서 논문의 변동성↔scheme 구조가 드러남.

#### 한계 (반드시 명시)
> 격차가 **WMAPE 0.2~0.5로 매우 작아** 통계적·실무적 유의성은 제한적입니다(66 시계열·13주 test). 방향성(저변동→ML, 고변동→SBC)이 논문과 일치한다는 데 의미가 있으며, **모든 데이터·base에서 동일하게 성립하지는 않습니다.**